This is the main enviorment of the project:
Object detection vs panoptic segmentation in adverse weather conditions

In [ ]:
import pandas  as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import sys
from ultralytics import YOLO
from PIL import Image

the device chosen and train the models is a rtx 4060M with 8gb VRAM

the dataset so far is ACDC.
other options:
WEDGE, DAWN, Cityscapes, COCO.

In [ ]:
data_path = 'C:\\desktop 2\\project\\datasets\\ACDC'
models_path = 'C:\\desktop 2\\project\\models'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Data orginizations, so far 70% training 20% validation and 10% test.

resolution hasn't been decided yet and the full data extraction isnt been implemented yet

In [ ]:
import glob
def dataFromPath(dataPath):
  path = data_path + dataPath
  images = {}
  for file in glob.glob(path):
      filename = file.split("\\")[-1]            # get the name of the .jpg file
      img = np.asarray(Image.open(file))        # read the image as a numpy array
      img =  np.array(img)/255 - 0.5
      images[filename] = img[:, :, :3]          # remove the alpha channel

  database = []

  # Populate train_data array
  for  (file_name, image) in images.items():
      id, pair_number, shoe_side, sex = file_name.split('_')
      person_index = int(id[1:]) - 1
      if shoe_side == 'left':
        shoe_side_index = 0
      else:
        shoe_side_index = 1
      pair_index = int(pair_number) - 1
      database.append([person_index,[pair_index, shoe_side_index],np.array(image)])

  database = sorted(database,key=lambda x: x[0])

  person_num = len(images)/6

  fin_data = np.zeros((int(person_num),3,2,224,224,3))
  for i,obj in enumerate(database):
    fin_data[i//6,obj[1][0],obj[1][1]] = obj[2]

  return fin_data

train_data = dataFromPath("data/train/*.jpg")
valid_data = train_data[90:]
train_data = train_data[:90]
test_data = dataFromPath("data/validation/*.jpg")

print("Shape of: Train - ", train_data.shape, "Validation - ", valid_data.shape)

NameError: name 'data_path' is not defined

models loading

In [ ]:
#Panoptic_segmentation_model = torch.hub.load(models_path, 'mask2former_r50_lsj_8x2_50e_coco_20220506_191028-8e96e88b.pth', pretrained=True)
#object_detection_model = torch.hub.load(models_path, 'yolov12l.pt', pretrained=True)
model = YOLO(r"c:\\desktop 2\\project\\models\\yolov12l.pt")

Training, this training function works with ADAM.
Reduce LT on Plateau scheduler
gradiant clipping
and checkpoints

In [ ]:
# you can change the signature and structure of this function as you please
# the code and comments below are only a suggestion to get you started
from torch.optim.lr_scheduler import ReduceLROnPlateau

def train_model(
    model,
    train_data,
    validation_data,
    batch_size=20,
    learning_rate=0.001,
    weight_decay=1e-3,
    epochs=30,
    checkpoint_path=None
):
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)
    
    best_valid_loss = float('inf')
    best_valid_acc = 0
    loss_data = []
    
    print(f"Training for Model {model.__class__.__name__}")
    print(f"n: {model.n}, Learning Rate: {learning_rate}, Weight Decay: {weight_decay}, "
          f"Epochs: {epochs}, Batch Size: {batch_size}")
    
    for epoch in range(epochs):
        model.train()
        
        positive_pair = generate_same_pair(train_data)
        negative_pair = generate_different_pair(train_data)
        
        reindex = np.random.permutation(len(negative_pair))
        negative_pair = negative_pair[reindex]
        reindex = np.random.permutation(len(positive_pair))
        positive_pair = positive_pair[reindex]
        
        for i in range(0, len(positive_pair), batch_size // 2):
            print(f"\r{epoch} {int(100 * i / len(positive_pair))}%", end='')
            
            pos_batch = positive_pair[i:i + batch_size // 2]
            neg_batch = negative_pair[i:i + batch_size // 2]
            
            if len(pos_batch) < batch_size // 2:
                continue
            
            batch = np.concatenate([pos_batch, neg_batch], axis=0)
            labels = np.concatenate([np.ones(len(pos_batch)), np.zeros(len(neg_batch))])
            
            reindex = np.random.permutation(len(batch))
            batch = batch[reindex]
            labels = labels[reindex]
            
            data = torch.Tensor(batch).permute(0, 3, 1, 2).to(device)
            labels = torch.Tensor(labels).long().to(device)
            
            # Training step
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, labels)
            loss.backward()
            
            # gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.5)
            
            optimizer.step()
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_positive_pair = generate_same_pair(validation_data)
            val_negative_pair = generate_different_pair(validation_data)
            
            val_inputs = np.concatenate([val_positive_pair, val_negative_pair], axis=0)
            val_inputs = torch.Tensor(val_inputs).permute(0, 3, 1, 2).to(device)
            num_val = len(val_positive_pair)
            val_labels = np.concatenate([np.ones(num_val), np.zeros(num_val)])
            val_labels = torch.Tensor(val_labels).long().to(device)
            
            pos_valid_accuracy, neg_valid_accuracy = get_accuracy(model, validation_data, batch_size=15, device=device)
            pos_train_accuracy, neg_train_accuracy = get_accuracy(model, train_data, batch_size=15, device=device)
            
            valid_loss = criterion(model(val_inputs), val_labels).item()
            valid_acc = 100 * (0.5 * (pos_valid_accuracy + neg_valid_accuracy))
            train_accuracy = 100 * (0.5 * (pos_train_accuracy + neg_train_accuracy))
            
            loss_data.append([epoch, valid_loss, loss.item()])
            
            print(
                f'\rEpoch [{epoch + 1}/{epochs}], '
                f'Val Loss: {valid_loss:.6f}, '
                f'Train Acc: {train_accuracy:.2f}, '
                f'Pos Val Acc: {pos_valid_accuracy*100:.2f}, '
                f'Neg Val Acc: {neg_valid_accuracy*100:.2f}, '
                f'Val Acc: {valid_acc:.2f}, '
                f'LR: {optimizer.param_groups[0]["lr"]:.8f}'
            )
            
            # Checkpoint
            if best_valid_acc == valid_acc and valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                name = model.__class__.__name__
                if checkpoint_path:
                    torch.save(model.state_dict(), 
                             f"{checkpoint_path}\\training_best_{name}.pk")
            
            if best_valid_acc < valid_acc:
                best_valid_acc = valid_acc
                best_valid_loss = valid_loss
                name = model.__class__.__name__
                if checkpoint_path:
                    torch.save(model.state_dict(), 
                             f"{checkpoint_path}\\training_best_{name}.pk")
            
            # Update learning rate
            scheduler.step(valid_loss)
    
    print(f'\nBest validation loss: {best_valid_loss:.6f}, '
          f'Best validation accuracy: {best_valid_acc:.2f}')
    
    model.load_state_dict(torch.load(f"{checkpoint_path}\\training_best_{name}.pk"))
    
    return best_valid_loss, best_valid_acc


evaluation so far will be PQ mAQ@50 and F1 for the models

In [ ]:
results = model.predict(
    source=str(data_path),  # accepts single file, folder, glob, webcam, etc.
    imgsz=640,              # forces 640×640 inference
    conf=0.25,              # tweak as desired
    save=True               # saves annotated images in runs/detect/predict
)

# Display results with bounding boxes and confidence percentages
for r in results:
    # Get the original image
    img = r.orig_img.copy()
    
    # Draw bounding boxes and labels with confidence
    if r.boxes is not None and len(r.boxes) > 0:
        for box in r.boxes:
            # Get box coordinates
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            
            # Get confidence and class
            conf = box.conf[0].cpu().numpy() * 100  # Convert to percentage
            cls_id = int(box.cls[0].cpu().numpy())
            cls_name = r.names[cls_id]
            
            # Draw rectangle (bounding box)
            color = (0, 255, 0)  # Green color
            thickness = 2
            img = cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
            
            # Create label with class name and confidence percentage
            label = f"{cls_name}: {conf:.1f}%"
            
            # Calculate text size for background rectangle
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.6
            font_thickness = 2
            (text_width, text_height), baseline = cv2.getTextSize(label, font, font_scale, font_thickness)
            
            # Draw background rectangle for text
            cv2.rectangle(img, (x1, y1 - text_height - 10), (x1 + text_width + 5, y1), color, -1)
            
            # Draw text
            cv2.putText(img, label, (x1 + 2, y1 - 5), font, font_scale, (0, 0, 0), font_thickness)
    
    # Convert BGR to RGB for display
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Display the image
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()


image 1/11 C:\desktop 2\project\datasets\ACDC\cloudy_0.jpg: 640x640 3 persons, 1 backpack, 31.6ms
image 2/11 C:\desktop 2\project\datasets\ACDC\cloudy_1.jpg: 640x640 5 persons, 35.7ms
image 3/11 C:\desktop 2\project\datasets\ACDC\cloudy_10.jpg: 640x640 3 persons, 1 handbag, 33.6ms
image 4/11 C:\desktop 2\project\datasets\ACDC\cloudy_2.jpg: 640x640 3 persons, 28.0ms
image 5/11 C:\desktop 2\project\datasets\ACDC\cloudy_3.jpg: 640x640 4 persons, 1 train, 29.3ms
image 6/11 C:\desktop 2\project\datasets\ACDC\cloudy_4.jpg: 640x640 1 train, 32.6ms
image 7/11 C:\desktop 2\project\datasets\ACDC\cloudy_5.jpg: 640x640 5 persons, 33.4ms
image 8/11 C:\desktop 2\project\datasets\ACDC\cloudy_6.jpg: 640x640 1 person, 28.4ms
image 9/11 C:\desktop 2\project\datasets\ACDC\cloudy_7.jpg: 640x640 2 persons, 1 train, 28.5ms
image 10/11 C:\desktop 2\project\datasets\ACDC\cloudy_8.jpg: 640x640 (no detections), 28.6ms
image 11/11 C:\desktop 2\project\datasets\ACDC\cloudy_9.jpg: 640x640 1 train, 29.1ms
Speed: 1

array([[[179, 166, 152],
        [179, 166, 152],
        [179, 166, 152],
        ...,
        [238, 223, 214],
        [237, 222, 213],
        [237, 222, 213]],

       [[179, 166, 152],
        [179, 166, 152],
        [179, 166, 152],
        ...,
        [237, 222, 213],
        [237, 222, 213],
        [237, 222, 213]],

       [[179, 166, 152],
        [179, 166, 152],
        [179, 166, 152],
        ...,
        [236, 221, 212],
        [236, 221, 212],
        [236, 221, 212]],

       ...,

       [[ 74, 105, 128],
        [ 75, 106, 129],
        [ 75, 106, 129],
        ...,
        [ 81, 111, 128],
        [ 81, 111, 128],
        [ 80, 110, 127]],

       [[ 73, 104, 127],
        [ 74, 105, 128],
        [ 74, 105, 128],
        ...,
        [ 81, 111, 128],
        [ 80, 110, 127],
        [ 80, 110, 127]],

       [[ 73, 104, 127],
        [ 73, 104, 127],
        [ 73, 104, 127],
        ...,
        [ 80, 110, 127],
        [ 80, 110, 127],
        [ 80, 110, 127]]

array([[[230, 230, 230],
        [230, 230, 230],
        [230, 230, 230],
        ...,
        [229, 225, 224],
        [229, 225, 224],
        [229, 225, 224]],

       [[230, 230, 230],
        [230, 230, 230],
        [230, 230, 230],
        ...,
        [229, 225, 224],
        [229, 225, 224],
        [229, 225, 224]],

       [[230, 230, 230],
        [230, 230, 230],
        [230, 230, 230],
        ...,
        [229, 225, 224],
        [229, 225, 224],
        [229, 225, 224]],

       ...,

       [[156, 165, 178],
        [156, 165, 178],
        [156, 165, 178],
        ...,
        [148, 163, 165],
        [148, 163, 165],
        [148, 163, 165]],

       [[156, 165, 178],
        [156, 165, 178],
        [156, 165, 178],
        ...,
        [147, 165, 166],
        [147, 165, 166],
        [147, 165, 166]],

       [[156, 165, 178],
        [156, 165, 178],
        [156, 165, 178],
        ...,
        [146, 164, 165],
        [146, 164, 165],
        [146, 164, 165]]

array([[[192, 191, 187],
        [192, 191, 187],
        [192, 191, 187],
        ...,
        [255,  42,   4],
        [255,  42,   4],
        [255,  42,   4]],

       [[192, 191, 187],
        [192, 191, 187],
        [192, 191, 187],
        ...,
        [255,  42,   4],
        [255,  42,   4],
        [255,  42,   4]],

       [[189, 190, 186],
        [189, 190, 186],
        [189, 190, 186],
        ...,
        [255,  62,  27],
        [255,  44,   6],
        [255,  42,   4]],

       ...,

       [[ 87,  88,  86],
        [ 89,  90,  88],
        [ 91,  92,  90],
        ...,
        [115, 116, 114],
        [124, 125, 123],
        [127, 128, 126]],

       [[ 90,  91,  89],
        [ 91,  92,  90],
        [ 91,  92,  90],
        ...,
        [118, 119, 117],
        [127, 128, 126],
        [130, 131, 129]],

       [[ 92,  93,  91],
        [ 92,  93,  91],
        [ 92,  93,  91],
        ...,
        [121, 122, 120],
        [130, 131, 129],
        [133, 134, 132]]

array([[[151, 133, 122],
        [151, 133, 122],
        [151, 133, 122],
        ...,
        [145, 127, 116],
        [145, 127, 116],
        [145, 127, 116]],

       [[151, 133, 122],
        [151, 133, 122],
        [150, 132, 121],
        ...,
        [145, 127, 116],
        [145, 127, 116],
        [145, 127, 116]],

       [[151, 133, 122],
        [150, 132, 121],
        [150, 132, 121],
        ...,
        [145, 127, 116],
        [145, 127, 116],
        [145, 127, 116]],

       ...,

       [[118, 141, 156],
        [118, 141, 156],
        [118, 141, 156],
        ...,
        [126, 138, 150],
        [126, 138, 150],
        [126, 138, 150]],

       [[118, 141, 156],
        [118, 141, 156],
        [118, 141, 156],
        ...,
        [127, 139, 151],
        [127, 139, 151],
        [127, 139, 151]],

       [[118, 141, 156],
        [118, 141, 156],
        [118, 141, 156],
        ...,
        [128, 140, 152],
        [128, 140, 152],
        [128, 140, 152]]

...

array([[[255, 236, 235],
        [255, 236, 235],
        [255, 236, 235],
        ...,
        [247, 215, 202],
        [247, 215, 202],
        [247, 215, 202]],

       [[255, 236, 235],
        [255, 236, 235],
        [255, 236, 235],
        ...,
        [247, 215, 202],
        [247, 215, 202],
        [247, 215, 202]],

       [[255, 236, 235],
        [255, 236, 235],
        [255, 236, 235],
        ...,
        [247, 215, 202],
        [247, 215, 202],
        [247, 215, 202]],

       ...,

       [[ 96,  89, 102],
        [ 96,  89, 102],
        [ 96,  89, 102],
        ...,
        [ 87,  84,  99],
        [ 87,  84,  99],
        [ 87,  84,  99]],

       [[ 95,  88, 101],
        [ 95,  88, 101],
        [ 95,  88, 101],
        ...,
        [ 87,  84,  99],
        [ 86,  83,  98],
        [ 86,  83,  98]],

       [[ 95,  88, 101],
        [ 95,  88, 101],
        [ 95,  88, 101],
        ...,
        [ 86,  83,  98],
        [ 86,  83,  98],
        [ 86,  83,  98]]

array([[[145, 132, 116],
        [143, 130, 114],
        [144, 131, 115],
        ...,
        [138, 129, 115],
        [139, 130, 116],
        [139, 130, 116]],

       [[145, 132, 116],
        [143, 130, 114],
        [144, 131, 115],
        ...,
        [138, 129, 115],
        [139, 130, 116],
        [139, 130, 116]],

       [[145, 132, 116],
        [143, 130, 114],
        [144, 131, 115],
        ...,
        [138, 129, 115],
        [139, 130, 116],
        [139, 130, 116]],

       ...,

       [[ 48,  39,  36],
        [ 48,  39,  36],
        [ 48,  39,  36],
        ...,
        [ 60,  49,  45],
        [ 60,  49,  45],
        [ 60,  49,  45]],

       [[ 47,  38,  35],
        [ 48,  39,  36],
        [ 48,  39,  36],
        ...,
        [ 60,  49,  45],
        [ 60,  49,  45],
        [ 60,  49,  45]],

       [[ 47,  38,  35],
        [ 47,  38,  35],
        [ 48,  39,  36],
        ...,
        [ 60,  49,  45],
        [ 60,  49,  45],
        [ 60,  49,  45]]

array([[[172, 144, 127],
        [172, 144, 127],
        [172, 144, 127],
        ...,
        [211, 180, 149],
        [211, 180, 149],
        [211, 180, 149]],

       [[172, 144, 127],
        [172, 144, 127],
        [172, 144, 127],
        ...,
        [211, 180, 149],
        [211, 180, 149],
        [211, 180, 149]],

       [[172, 144, 127],
        [172, 144, 127],
        [172, 144, 127],
        ...,
        [211, 180, 149],
        [211, 180, 149],
        [211, 180, 149]],

       ...,

       [[ 85,  75,  68],
        [ 85,  75,  68],
        [ 85,  75,  68],
        ...,
        [ 95,  83,  73],
        [ 95,  83,  73],
        [ 95,  83,  73]],

       [[ 96,  86,  79],
        [ 94,  84,  77],
        [ 91,  81,  74],
        ...,
        [ 95,  83,  73],
        [ 95,  83,  73],
        [ 95,  83,  73]],

       [[ 94,  84,  77],
        [ 92,  82,  75],
        [ 89,  79,  72],
        ...,
        [ 95,  83,  73],
        [ 95,  83,  73],
        [ 95,  83,  73]]

array([[[244, 232, 220],
        [244, 232, 220],
        [244, 232, 220],
        ...,
        [158, 145, 129],
        [157, 144, 128],
        [157, 144, 128]],

       [[244, 232, 220],
        [244, 232, 220],
        [244, 232, 220],
        ...,
        [157, 144, 128],
        [157, 144, 128],
        [157, 144, 128]],

       [[244, 232, 220],
        [244, 232, 220],
        [244, 232, 220],
        ...,
        [157, 144, 128],
        [156, 143, 127],
        [156, 143, 127]],

       ...,

       [[  5,   5,   5],
        [  5,   5,   5],
        [  5,   5,   5],
        ...,
        [  8,   8,   8],
        [  8,   8,   8],
        [  8,   8,   8]],

       [[  5,   5,   5],
        [  5,   5,   5],
        [  5,   5,   5],
        ...,
        [  9,   9,   9],
        [  9,   9,   9],
        [  9,   9,   9]],

       [[  5,   5,   5],
        [  5,   5,   5],
        [  5,   5,   5],
        ...,
        [  9,   9,   9],
        [  9,   9,   9],
        [  9,   9,   9]]

array([[[163, 159, 158],
        [163, 159, 158],
        [163, 159, 158],
        ...,
        [119, 107, 105],
        [119, 107, 105],
        [119, 107, 105]],

       [[163, 159, 158],
        [163, 159, 158],
        [163, 159, 158],
        ...,
        [119, 107, 105],
        [119, 107, 105],
        [119, 107, 105]],

       [[163, 159, 158],
        [163, 159, 158],
        [163, 159, 158],
        ...,
        [119, 107, 105],
        [119, 107, 105],
        [119, 107, 105]],

       ...,

       [[ 24,  26,  27],
        [ 24,  26,  27],
        [ 24,  26,  27],
        ...,
        [ 31,  33,  33],
        [ 31,  33,  33],
        [ 31,  33,  33]],

       [[ 24,  26,  27],
        [ 24,  26,  27],
        [ 24,  26,  27],
        ...,
        [ 30,  32,  32],
        [ 30,  32,  32],
        [ 30,  32,  32]],

       [[ 24,  26,  27],
        [ 24,  26,  27],
        [ 24,  26,  27],
        ...,
        [ 30,  32,  32],
        [ 30,  32,  32],
        [ 29,  31,  31]]